# UC-ASSET-2 — Asset References and Deletion Protection

**Persona:** Platform Administrator

**Goal:** List all references attached to an asset, identify which references
block hard-deletion (`cascade_delete=false`), remove those blocking refs, then
issue a force hard-delete and confirm the asset is gone (HTTP 404).

**Prerequisites:**
- A target asset exists in `CATALOG_ID` (set `ASSET_ID` in env)
- At least one reference with `cascade_delete=false` is registered
  (e.g. a DuckDB table registered by the storage driver on attach)
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`,
`CATALOG_ID`, `ASSET_ID`

**Memory refs:**
- `project_geoid_storage_drivers` — `AssetReference`, `cascade_delete`, ref guard
- `modules/catalog/CLAUDE.md` — `AssetReferencedError` → HTTP 409 guard logic

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL    = os.environ["DYNASTORE_BASE_URL"]
ADMIN_TOKEN = os.environ["DYNASTORE_ADMIN_TOKEN"]
CATALOG_ID  = os.environ.get("CATALOG_ID", "demo-catalog")
ASSET_ID    = os.environ["ASSET_ID"]

headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)
print(f"Connected to {BASE_URL}")
print(f"Catalog: {CATALOG_ID}  Asset: {ASSET_ID}")

In [ ]:
# List all references on the asset
r = client.get(f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}/references")
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

refs = r.json()  # List[AssetReference]
print(f"\nTotal references: {len(refs)}")
for ref in refs:
    print(
        f"  ref_type={ref.get('ref_type'):<25}  "
        f"ref_id={ref.get('ref_id'):<30}  "
        f"cascade_delete={ref.get('cascade_delete')}"
    )

In [ ]:
# Identify blocking references: cascade_delete=false means hard-delete is blocked
blocking = [r for r in refs if not r.get("cascade_delete", True)]
cascading = [r for r in refs if r.get("cascade_delete", True)]

print(f"Blocking refs   : {len(blocking)}  (cascade_delete=false — must be removed manually)")
print(f"Cascading refs  : {len(cascading)}  (cascade_delete=true  — removed automatically by PG trigger)")

for b in blocking:
    print(f"  BLOCKING: ref_type={b.get('ref_type')}  ref_id={b.get('ref_id')}")

In [ ]:
# Attempt hard-delete while blocking refs remain → expect HTTP 409
if blocking:
    r = client.delete(
        f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}",
        params={"force": "true"},
    )
    print(r.status_code, r.text[:400])
    assert r.status_code == 409, (
        f"Expected 409 Conflict (blocking refs present), got {r.status_code}: {r.text}"
    )
    detail = r.json().get("detail", {})
    print(f"\n409 confirmed — blocking_references count: {len(detail.get('blocking_references', []))}")
    print("Hard-deletion correctly rejected while blocking references remain.")
else:
    print("No blocking references found — skipping 409 assertion (asset has no non-cascading refs).")

In [ ]:
# Remove each blocking reference
# WARNING: only call this after confirming the referencing entity has been cleaned up.
# Removing a blocking reference allows hard-deletion even if the table/driver
# that created it has not been dropped.

for ref in blocking:
    ref_type = ref["ref_type"]
    ref_id   = ref["ref_id"]
    r = client.delete(
        f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}/references/{ref_type}/{ref_id}"
    )
    print(f"DELETE ref  ref_type={ref_type}  ref_id={ref_id}  → {r.status_code}")
    assert r.status_code == 204, (
        f"Expected 204 removing reference {ref_type}/{ref_id}, got {r.status_code}: {r.text}"
    )

print(f"\nAll {len(blocking)} blocking reference(s) removed.")

In [ ]:
# Hard-delete the asset now that all blocking refs are removed
r = client.delete(
    f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}",
    params={"force": "true", "propagate": "true"},
)
print(r.status_code, r.text[:200])
assert r.status_code == 204, (
    f"Expected 204 after blocking refs removed, got {r.status_code}: {r.text}"
)
print(f"Asset '{ASSET_ID}' hard-deleted successfully.")

In [ ]:
# Confirm asset is gone — GET must return 404
r = client.get(f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}")
print(r.status_code, r.text[:200])
assert r.status_code == 404, (
    f"Expected 404 after hard-delete, got {r.status_code}: {r.text}"
)
print(f"Confirmed: asset '{ASSET_ID}' is no longer accessible (404).")

## Edge cases

### Cascading references (cascade_delete=true)

References with `cascade_delete=true` are **informational**: they record which
entities hold a logical dependency on the asset but do not block hard-deletion.
When `DELETE ... ?force=true` executes, the PostgreSQL `ON DELETE CASCADE` trigger
on `asset_references` removes these rows automatically.  No manual cleanup needed.

Use case: a STAC `collection` reference is registered with `cascade_delete=true`
so that dropping the collection also drops the reference without blocking the
asset's own lifecycle.

In [ ]:
# cascade_delete=True refs — documentation cell
#
# When a storage driver (e.g. DuckDB) attaches a collection it registers:
#   AssetReference(
#       ref_type="collection",          # CoreAssetReferenceType.COLLECTION
#       ref_id=collection_id,
#       cascade_delete=True,            # non-blocking
#   )
#
# On hard-delete the PG trigger removes this row automatically:
#   DELETE FROM asset_references WHERE asset_id = $1 AND cascade_delete = TRUE;
#
# The deletion guard only inspects cascade_delete=FALSE rows (partial index).
# No action needed from the client for cascading references.

cascade_docs = [r for r in refs if r.get("cascade_delete", True)]
print(f"Cascading (auto-removed) references on this asset: {len(cascade_docs)}")
for c in cascade_docs:
    print(f"  ref_type={c.get('ref_type')}  ref_id={c.get('ref_id')}  — removed by PG trigger")

In [ ]:
# Schema note: known FK constraint between asset_references and assets
#
# asset_references.asset_id references assets.asset_id
# Both tables live in the tenant schema ({catalog_id}.assets, {catalog_id}.asset_references).
#
# KNOWN ISSUE: if the assets table uses a composite PK (asset_id, catalog_id)
# but asset_references only FKs on asset_id, cross-catalog ref lookups may
# return stale rows from a different catalog's asset with the same asset_id.
#
# Mitigation: all API endpoints scope queries to catalog_id in the WHERE clause.
# Ensure asset_id values are globally unique (e.g. use UUID or namespaced IDs)
# when operating multi-catalog platforms to avoid ambiguity.

print("Schema note acknowledged: asset_references FK scoped to catalog_id at query level.")
print("Use namespaced or UUID asset_ids to prevent cross-catalog collisions.")
client.close()